# MNIST Inference with OpenEye Toolkit and ONNX
This notebook demonstrates how to use the OpenEye Toolkit to run inference 
on the MNIST dataset using an ONNX model.

It covers training a simple neural network, exporting it to ONNX format, and
then quantizing it using the ONNX Runtime and the OpenEye Toolkit to simulate
inference on an image of the test dataset.

In [ ]:
# check if required packages are installed, if not ask to install them using pip
import importlib
import subprocess
import sys

required_packages = [
    "torch",
    "torchvision",
    "onnx",
    "onnxruntime",
    "onnxscript",
    "onnxoptimizer"
]

all_packages_installed = True
for package in required_packages:
    if importlib.util.find_spec(package) is None:
        all_packages_installed = False
        print(f"{package} is not installed. Should I install it for you? (y/n)")
        choice = input().lower()
        if choice == 'y':
            subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        else:
            print(f"Please install {package} manually.")

if all_packages_installed:
    print("All required packages are already installed.")

In [ ]:
# import necessary libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from mnist_conv_net import SimpleMNISTConvNet

# additional imports for ONNX handling
import onnx
import onnxruntime as ort
import onnxoptimizer

import numpy as np

In [ ]:
from mnist_conv_net import SimpleMNISTConvNet, load_simple_mnist_model

model = SimpleMNISTConvNet()

load_simple_mnist_model('mnist_unquantized_model.pth', model)

In [ ]:
# export the model to ONNX format
dummy_input = torch.randn(1, 1, 28, 28)  # batch size 1, 1 channel, 28x28 image
onnx_model_path = "mnist_unquantized_model.onnx"
torch.onnx.export(model, dummy_input, onnx_model_path, input_names=['input'], output_names=['output'])
print(f"Model exported to ONNX format at: {onnx_model_path}")

In [ ]:
# load the ONNX model and verify
onnx_model = onnx.load(onnx_model_path)
onnx.checker.check_model(onnx_model)
print("ONNX model is valid.")


In [ ]:
# Example pruning function: zero out weights below a certain threshold (magnitude-based pruning)
def prune_model(model, threshold=0.01):
    for layer in model.graph.node:
        # Iterate through weights in each layer
        for attr in layer.attribute:
            if attr.name == 'weights':  # Assuming 'weights' attribute stores weights
                weights = np.frombuffer(attr.tensor.raw_data, dtype=np.float32)
                weights[abs(weights) < threshold] = 0  # Zero out weights below threshold
                attr.tensor.raw_data = weights.tobytes()  # Update the weights
    return model

# Prune the ONNX model
pruned_onnx_model = prune_model(onnx_model, threshold=0.01)

In [ ]:
print(pruned_onnx_model)

In [ ]:
# Save the pruned model
pruned_model_path = "mnist_pruned_model.onnx"
onnx.save(pruned_onnx_model, pruned_model_path)
print(f"Pruned model saved at {pruned_model_path}")

In [ ]:
# Optimize the pruned model
print("Available optimization passes:")
print(onnxoptimizer.get_fuse_and_elimination_passes())

passes = onnxoptimizer.get_fuse_and_elimination_passes()
optimized_model = onnxoptimizer.optimize(pruned_onnx_model, passes)

# Save the optimized pruned model
optimized_model_path = "mnist_optimized_pruned_model.onnx"
onnx.save(optimized_model, optimized_model_path)
print(f"Optimized pruned model saved at {optimized_model_path}")

In [ ]:
def measure_inference_time(model_path, input_shape, num_runs=100):
    import time
    session = ort.InferenceSession(model_path)
    input_name = session.get_inputs()[0].name
    dummy_input = np.random.rand(*input_shape).astype(np.float32)
    
    # Warm-up
    session.run(None, {input_name: dummy_input})
    
    # Measure inference time
    start_time = time.time()
    for _ in range(num_runs):
        session.run(None, {input_name: dummy_input})
    avg_time = (time.time() - start_time) / num_runs
    print(f"Average inference time for {model_path}: {avg_time:.10f} seconds")

# Measure performance of original, pruned, and optimized models
measure_inference_time(onnx_model_path, (1, 1, 28, 28))
measure_inference_time(pruned_model_path, (1, 1, 28, 28))
measure_inference_time(optimized_model_path, (1, 1, 28, 28))


In [ ]:
print(onnx_model)

In [ ]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Transform: Converts images to PyTorch tensors and normalizes them
# Normalization: (value - mean) / standard deviation
# This helps the neural network learn better
transform = transforms.Compose([
    transforms.ToTensor(),                    # Image to tensor (0-255 -> 0-1)
    transforms.Normalize((0.1307,), (0.3081,))  # Normalization with MNIST statistics
])

train_dataset = datasets.MNIST(
    root='./data',           # Storage location
    train=True,              # Training data
    download=True,           # Download if not present
    transform=transform      # Apply transformations
)

test_dataset = datasets.MNIST(
    root='./data',           # Storage location
    train=False,             # Test data
    download=True,          # Download if not present
    transform=transform     # Apply transformations
)


In [ ]:
from onnxruntime.quantization import CalibrationDataReader, quantize_static, QuantType

class MNISTCalibrationDataReader(CalibrationDataReader):
    def __init__(self, dataset, batch_size=1):
        self.dataset = dataset
        self.batch_size = batch_size
        self.data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
        self.iterator = iter(self.data_loader)
        self.input_name = None

    def get_next(self):
        try:
            batch = next(self.iterator)
            images, _ = batch  # We only need images for calibration
            if self.input_name is None:
                # Get input name from the first batch
                session = ort.InferenceSession(optimized_model_path)
                self.input_name = session.get_inputs()[0].name
            return {self.input_name: images.numpy()}
        except StopIteration:
            return None

calibration_data_reader = MNISTCalibrationDataReader(train_dataset, batch_size=1)


In [ ]:

quantized_model_path = "mnist_quantized_model.onnx"

# Perform quantization on the optimized model
quantize_static(
    model_input=optimized_model_path,
    model_output=quantized_model_path,
    calibration_data_reader=calibration_data_reader,
    activation_type=QuantType.QInt8,
    weight_type=QuantType.QInt8  # You can also use QuantType.QInt8 if supported
)

print(f"Quantized model saved at {quantized_model_path}")

In [ ]:
import time

def benchmark_model(session, input_data):
    input_name = session.get_inputs()[0].name
    
    # Warm-up run (to avoid first-run overhead)
    session.run(None, {input_name: input_data})
    
    # Measure inference time
    start_time = time.time()
    for _ in range(100):  # Run multiple iterations for more stable results
        session.run(None, {input_name: input_data})
    end_time = time.time()
    
    avg_inference_time = (end_time - start_time) / 100
    print(f"Average inference time: {avg_inference_time:.6f} seconds")
    return avg_inference_time

# Load original and quantized models
original_session = ort.InferenceSession(optimized_model_path)
quantized_session = ort.InferenceSession(quantized_model_path)

# Benchmark using a sample input
sample_input = np.random.rand(1, 1, 28, 28).astype(np.float32)  # Replace with actual preprocessed input
print("Original model:")
benchmark_model(original_session, sample_input)

print("Quantized model:")
benchmark_model(quantized_session, sample_input)

In [ ]:
import re
print(onnx.__version__, " opset=", onnx.defs.onnx_opset_version())

In [ ]:
print(onnx_model)

quantized_model = onnx.load(quantized_model_path)

print(quantized_model)

In [ ]:

def shape2tuple(shape):
    return tuple(getattr(d, 'dim_value', 0) for d in shape.dim)

# in a more nicely format
print('** inputs **')
for obj in onnx_model.graph.input:
    print("name=%r dtype=%r shape=%r" % (
        obj.name, obj.type.tensor_type.elem_type,
        shape2tuple(obj.type.tensor_type.shape)))


In [ ]:
# in a more nicely format
print('** outputs **')
for obj in quantized_model.graph.output:
    print("name=%r dtype=%r shape=%r" % (
        obj.name, obj.type.tensor_type.elem_type,
        shape2tuple(obj.type.tensor_type.shape)))

In [ ]:
from onnx import numpy_helper

In [ ]:
# in a more nicely format
print('** nodes **')
for node in quantized_model.graph.node:
    print("name=%r type=%r input=%r output=%r" % (
        node.name, node.op_type, node.input, node.output))
    # print type of weight inputs

print('\n** initializers **')
for initializer in quantized_model.graph.initializer:
    n = numpy_helper.to_array(initializer)
    print("initializer name=%r shape=%r dtype=%r" % (
        initializer.name,
        n.shape,
        n.dtype))

In [ ]:
# find node with name "input_QuantizeLinear"
nodes_dict = {node.name: node for node in quantized_model.graph.node}
initializers_dict = {init.name: init for init in quantized_model.graph.initializer}
input_quantize_linear_node = nodes_dict.get("input_QuantizeLinear", None)
print(input_quantize_linear_node)

for input in input_quantize_linear_node.input:
    for initializer in quantized_model.graph.initializer:
        if initializer.name == input:
            n = numpy_helper.to_array(initializer)
            print("initializer name=%r shape=%r dtype=%r values=%r" % (
                initializer.name,
                n.shape,
                n.dtype,
                n))
            

In [ ]:
# get a sample image from the test dataset
sample_image, sample_label = test_dataset[0]
sample_image = sample_image.numpy()  # convert to numpy array
print("Sample image shape:", sample_image.shape)

# as a test, execute the onnx runtime model on this image and 
# add first dimension for batch size
sample_input = sample_image[np.newaxis, :, :, :]
ort_sess = ort.InferenceSession(onnx_model_path)
results = ort_sess.run(None, {ort_sess.get_inputs()[0].name: sample_input})
print("Running ONNX Runtime on quantized model...")
print(ort_sess.get_outputs()[0].name)
print(results)


# now, we use the quantized model
ort_sess = ort.InferenceSession(quantized_model_path)
results = ort_sess.run(None, {ort_sess.get_inputs()[0].name: sample_input})
print("Running ONNX Runtime on quantized model...")
print(ort_sess.get_outputs()[0].name)
print(results)


In [ ]:
# now, we want to store all intermediate results of the model execution on the sample image
intermediate_results = {}
for node in quantized_model.graph.node:
    for output in node.output:
        intermediate_results[output] = None
from onnx.reference import ReferenceEvaluator
ref_evaluator = ReferenceEvaluator(quantized_model)
inputs = {ref_evaluator.model.graph.input[0].name: sample_input}
all_outputs = ref_evaluator.run(inputs)
for output_name, output_value in zip(ref_evaluator.model.graph.output, all_outputs):
    print(f"Output {output_name.name}: shape={output_value.shape}, dtype={output_value.dtype}")
    intermediate_results[output_name.name] = output_value
# print intermediate results
for name, value in intermediate_results.items():
    if value is not None:
        print(f"Intermediate result {name}: shape={value.shape}, dtype={value.dtype}")
sample_input = sample_image[np.newaxis, :, :, :]
ort_sess = ort.InferenceSession(onnx_model_path)
x, y = test_dataset[0][0], test_dataset[0][1]
print(x.shape, y)

In [ ]:
for node in quantized_model.graph.node:
    if node.op_type in ['QuantizeLinear', 'DequantizeLinear']:
        continue
    print("name=%r type=%r input=%r output=%r" % (
        node.name, node.op_type, node.input, node.output))
    for input_name in node.input:
        for initializer in quantized_model.graph.initializer:
            initializer_name = initializer.name.strip("_quantized")
            if initializer_name == input_name:
                n = numpy_helper.to_array(initializer)
                print("Node %r has weight input %r with shape=%r dtype=%r and value %r" % (
                    node.name,
                    input_name,
                    n.shape,
                    n.dtype,
                    n))
    